In [ ]:
# bsnn_simulation.ipynb

import copy
import json
import os
import platform
from fractions import Fraction

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn import svm
from sklearn.svm import SVC
from common_functions import *
from data_loader import *



def main(N, test_portion, num_epochs, n_iterations, data_type,
          folder_path, modes, depth,
          X_data, y_data, X_data_train, X_data_test, y_data_train, y_data_test, num_photons, numerator, denominator, init_state=None, state_name=None):
    print("DEBUG main() got N =", N)

    

    global model

    device = torch.device("cpu")

    start_jj = 0
    state_name, init_state = get_leftmost_state(modes, num_photons)


    



    device = torch.device("cpu")
    results_file_path = os.path.join(
        folder_path,f"accuracies_N{N}_modes{modes}_depth{depth}_photons{num_photons}_state_{state_name}_epochs{num_epochs-1}.json"
        )
    if os.path.exists(results_file_path):
        with open(results_file_path, "r") as f:
            final_results = json.load(f)
        existing_count = len(final_results["quantum"].get("q_prime_acc_test", []))
        start_jj = len(final_results["quantum"].get("q_prime_acc_test", []))  
    
    for jj in range(start_jj, n_iterations):
        seed_model = SEED + jj

        set_all_seeds(seed_model)
        rng_state = get_rng_state()
        if data_type in ('cifar10','MNIST','MNISTresized','fashionMNIST','MNIST01','MNIST17',
                 'fashionMNIST_Tshirt_dress','organMNISTa','organMNISTb'):

            idx_tr, idx_te = get_or_make_mnist_sep_indices(
                folder_path=folder_path,
                data_type=data_type,
                N=N,
                test_portion=test_portion,
                jj=jj,
                n_train_total=len(X_data_train),
                n_test_total=len(X_data_test),
                seed_base=SEED
            )

            (X_raw, X_raw_train, X_raw_test, y_labels, y_train, y_test,
            subset_indexes_train, subset_indexes_test, global_indexes_train, global_indexes_test) = \
                build_from_mnist_separate(X_data_train, y_data_train, X_data_test, y_data_test, idx_tr, idx_te)
        else:
            

            total_len = len(X_data)
            idx_subset, idx_train_global, idx_test_global = get_or_make_split_indices(
                folder_path=folder_path,
                data_type=data_type,
                N=N,
                test_portion=test_portion,
                jj=jj,
                total_len=total_len,
                seed_base=SEED
            )

            (X_raw, X_raw_train, X_raw_test, y_labels, y_train, y_test,
            subset_indexes_train, subset_indexes_test, global_indexes_train, global_indexes_test) = \
                build_from_global_split(X_data, y_data, idx_subset, idx_train_global, idx_test_global)



        
    
        if num_photons > modes:
            continue
        print('modes:', modes,' depth:',depth)
        print('photons: ', num_photons)


        
        if os.path.exists(results_file_path):
            with open(results_file_path, "r") as f:
                final_results = json.load(f)
            existing_count = len(final_results["quantum"].get("q_prime_acc_test", []))
            print('existing_count:',existing_count)
            if existing_count >= 5:
                continue   
        else:
            os.makedirs(os.path.dirname(results_file_path), exist_ok=True)

        num_train = int((1 - test_portion) * N)
        z_ij = zij_calc(num_train,y_train)
        num_classes = len(np.unique(y_labels))
        _, num_circ_params = pad_input(X_raw, modes, depth, init_state)
        if num_classes > 2:
            classifier = OneVsRestClassifier(SVC(kernel='precomputed'))
        else:
            classifier = svm.SVC(kernel='precomputed', verbose=False)

        if num_photons == 5:
            models_group = ['quantum','linear', 'sigmoid', 'CNN'] 
        else:
            models_group = ['quantum']


        

        for model_type in models_group: 
            set_rng_state(rng_state)   # <-- makes model_type order irrelevant
            print(model_type, torch.rand(3)[:3], np.random.rand(3))

            if model_type in {'linear', 'poly', 'rbf', 'sigmoid'} and num_photons == 5:
                # Direct kernel usage
                classifier = SVC(kernel=model_type, degree=3, coef0=1, gamma='scale')  
                classifier.fit(X_raw_train, y_train)
                y_test_pred = classifier.predict(X_raw_test)
                acc = accuracy_score(y_test, y_test_pred)
                results_file_path = os.path.join(
                folder_path,f"accuracies_N{N}_modes{modes}_depth{depth}_photons{num_photons}_state_{state_name}_epochs{num_epochs-1}.json"
                )
                if os.path.exists(results_file_path):
                    with open(results_file_path, "r") as f:
                        final_results = json.load(f)
                else:
                    final_results = {}
                                    # Ensure all model types exist
                default_template = {
                    'c_acc_test': [], 'c_prime_acc_test': [], 'c_prime_acc_train': []
                }
                if 'quantum' not in final_results:
                    final_results['quantum'] = {**default_template, 'q_acc_test': [], 'q_prime_acc_test': [], 'q_prime_acc_train': [], 'q_loss': []}
                for key in ['coherent', 'linear', 'poly', 'rbf', 'sigmoid', 'CNN']:
                    if key not in final_results:
                        final_results[key] = copy.deepcopy(default_template)
                final_results[model_type]['c_prime_acc_test'].append(acc)
                print(model_type, acc)
                with open(results_file_path, "w") as f:
                    json.dump(final_results, f, indent=4)
            else:
                print(model_type, 'iter:',jj)
                nn_in_dim = len(X_raw[0])  
                nn_out_dim = num_circ_params

                if model_type == 'quantum':
                    model = QuantumKernelNN(
                        input_dim=nn_in_dim,
                        output_dim=nn_out_dim,
                        init_state=init_state,
                        modes=modes,
                        depth=depth,
                        block_a=128,
                        block_b=128,
                        chunk_subsets=1<<13
                    )

                elif model_type == 'CNN':
                    num_classes = len(np.unique(y_labels))
                    model = SimpleLinearModel(input_dim=nn_in_dim, mid_dim=num_circ_params, output_dim=num_classes)
                model = model.to(device)
                optimizer = optim.Adam(model.parameters(), lr=0.001)
                if model_type == 'CNN':
                    criterion = nn.CrossEntropyLoss()
                else:
                    criterion = nn.MSELoss()
                X_tensor = torch.stack([x.clone().detach().to(device).requires_grad_(True) for x in X_raw_train])
                y_tensor = y_train.to(dtype=torch.long, device=device)
                # after X_tensor is defined
                K = min(32, len(X_raw_train))  # small fixed set
                X_probe = torch.stack([x.to(device) for x in X_raw_train[:K]])
                



                # Training loop
                for epoch in range(num_epochs):
                    results_file_path = os.path.join(
                    folder_path,f"accuracies_N{N}_modes{modes}_depth{depth}_photons{num_photons}_state_{state_name}_epochs{epoch}.json"
                    )
                    if os.path.exists(results_file_path):
                        with open(results_file_path, "r") as f:
                            final_results = json.load(f)
                    else:
                        final_results = {}

                    # Ensure all model types exist
                    default_template = {
                        'c_acc_test': [], 'c_prime_acc_test': [], 'c_prime_acc_train': [], 'c_loss': []
                    }
                    if 'quantum' not in final_results:
                        final_results['quantum'] = {**default_template, 'q_acc_test': [], 'q_prime_acc_test': [], 'q_prime_acc_train': [], 'q_loss': []}
                    for key in ['coherent', 'linear', 'poly', 'rbf', 'sigmoid', 'CNN']:
                        if key not in final_results:
                            final_results[key] = copy.deepcopy(default_template)
                    optimizer.zero_grad()
                    
                    if model_type == 'CNN':
                        if y_tensor.min() < 0:
                            y_tensor = (y_tensor + 1) // 2  # Convert -1 to 0, +1 stays 1
                        model.train()
                        optimizer.zero_grad()
                        outputs = model(X_tensor)
                        loss = criterion(outputs, y_tensor)
                        loss.backward()
                        optimizer.step()

                        _, predicted = torch.max(outputs, 1)
                        accuracy = (predicted == y_tensor).float().mean().item()
                        final_results[model_type]['c_prime_acc_train'].append(accuracy)
                        
                        # Evaluation
                        model.eval()
                        with torch.no_grad():
                            X_test_tensor = torch.stack([x.to(device) for x in X_raw_test])
                            # y_test_tensor = y_test.to(dtype=torch.long, device=device)
                            y_test_tensor = y_test.to(dtype=torch.long, device=device)
                            if y_test_tensor.min() < 0:
                                y_test_tensor = (y_test_tensor + 1) // 2

                            outputs_test = model(X_test_tensor)
                            _, predicted_test = torch.max(outputs_test, 1)
                            test_acc = (predicted_test == y_test_tensor).float().mean().item()
                            final_results[model_type]['c_prime_acc_test'].append(test_acc)
                            # print(f"Test Accuracy: {test_acc:.4f}")
                    elif model_type == 'quantum':
                        X_prime_train, Gram_prime_train = model(X_tensor)  # Calls forward() to get: 1 X_prime 
                        # print(X_tensor)
                        # print(Gram_prime_train)
                        loss = criterion(Gram_prime_train, z_ij)
                        loss.backward()
                        optimizer.step()
                        X_prime_train = X_prime_train.detach()  # Stops gradient tracking
                        Gram_prime_train = Gram_prime_train.detach()
                        X_prime, X_prime_test = convert_data(N,X_raw_test, subset_indexes_test, subset_indexes_train , X_prime_train,model)
                    
                    if epoch % 10 == 0:
                        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}")



                    if model_type in {'quantum'}:
                        quant_gram_prime, classic_gram_prime = photonic_Gram_calculator(X_prime, modes, depth, init_state)
                        Gram_prime_q_train, Gram_prime_c_train = photonic_Gram_calculator(X_prime_train, modes, depth, init_state)
                        if model_type == 'quantum':
                            if epoch == num_epochs-1:
                                print("accuracy",SVM_acc_test(quant_gram_prime, classifier, subset_indexes_train , subset_indexes_test, y_train, y_test)[0])

                            final_results['quantum']['q_prime_acc_test'].append(SVM_acc_test(quant_gram_prime, classifier, subset_indexes_train , subset_indexes_test, y_train, y_test)[0])
                            if 'q_loss' not in final_results['quantum']:
                                final_results['quantum']['q_loss'] = []
                            final_results['quantum']['q_loss'].append(loss.item())
                            final_results['quantum']['q_prime_acc_train'].append(SVM_acc_train(X_prime_train, y_train, Gram_prime_q_train, classifier)[0])


                    with open(results_file_path, "w") as f:
                        json.dump(final_results, f, indent=4)

                    if epoch == num_epochs - 1 and model_type != 'CNN':
                        save_dir = os.path.join(os.path.dirname(results_file_path), "large_data")
                        os.makedirs(save_dir, exist_ok=True)

                        large_data_prefix = f"N{N}_modes{modes}_depth{depth}_photons{num_photons}_state_{state_name}_{model_type}_epochs{epoch}"
                        save_path = os.path.join(save_dir, f"data_{large_data_prefix}.npy")

                        # ---- Convert to compact numpy types ----
                        y_train_np = (y_train.cpu().numpy() if isinstance(y_train, torch.Tensor) else np.array(y_train)).astype(np.int64)
                        y_test_np  = (y_test.cpu().numpy()  if isinstance(y_test, torch.Tensor)  else np.array(y_test)).astype(np.int64)

                        X_prime_train_np = (X_prime_train if isinstance(X_prime_train, np.ndarray)
                                            else X_prime_train.cpu().detach().numpy()).astype(np.float32)
                        X_prime_test_np  = (X_prime_test if isinstance(X_prime_test, np.ndarray)
                                            else X_prime_test.cpu().detach().numpy()).astype(np.float32)

                        # Indices back to the ORIGINAL dataset split returned by prepare_data(data_type)
                        # For non-MNIST branch, we set them above to subset_indexes_*.
                        gtr = np.array(global_indexes_train, dtype=np.int64)
                        gte = np.array(global_indexes_test, dtype=np.int64)


                        is_mnist_like = data_type in (
                            'cifar10','MNIST','MNISTresized','fashionMNIST','MNIST01','MNIST17',
                            'fashionMNIST_Tshirt_dress','organMNISTa','organMNISTb'
                        )

                        record = {
                            # --- data (compact) ---
                            "X_prime_train": X_prime_train_np,
                            "X_prime_test":  X_prime_test_np,
                            "y_train": y_train_np,
                            "y_test":  y_test_np,

                            # --- indices into ORIGINAL arrays returned by prepare_data() ---
                            # MNIST-like: train indices index X_train/y_train; test indices index X_test/y_test
                            # non-MNIST:  both index X_data/y_data
                            "global_idx_train": gtr,
                            "global_idx_test":  gte,
                            "index_reference": (
                                "prepare_data: X_train/X_test (separate)"
                                if is_mnist_like else
                                "prepare_data: X_data"
                            ),

                            # --- experiment identity ---
                            "data_type": str(data_type),
                            "N": int(N),
                            "test_portion": float(test_portion),
                            "modes": int(modes),
                            "depth": int(depth),
                            "num_photons": int(num_photons),
                            "state_name": str(state_name),
                            "model_type": str(model_type),
                            "epoch": int(epoch),
                            "iteration": int(jj),

                            # --- RNG / reproducibility metadata ---
                            "seed_base": int(SEED),
                            "seed_effective": int(SEED + jj),

                            # Helpful when debugging order effects:
                            "rng_reset_per_model_type": True,  # you do set_rng_state(rng_state) inside the model_type loop

                            # Optional: deterministic settings snapshot
                            "torch_num_threads": int(torch.get_num_threads()),
                            "torch_deterministic_algorithms": True,
                            "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
                            "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),

                            # --- split cache file used (so you can replay exact split) ---
                            "split_cache_file": (
                                split_cache_path_mnist(folder_path, data_type, N, test_portion, jj)
                                if is_mnist_like else
                                split_cache_path(folder_path, data_type, N, test_portion, jj)
                            ),
                        }

                        # Append into a dict-of-runs file
                        if os.path.exists(save_path):
                            all_data = np.load(save_path, allow_pickle=True).item()
                        else:
                            all_data = {}

                        all_data[len(all_data)] = record
                        np.save(save_path, all_data)
         



for data_type in ['spambase']:
    X_data, y_data, X_train, X_test, y_train, y_test = prepare_data(data_type)


    if data_type in {'MNIST', 'fashionMNIST'}:
        N = 5000
        SEED = 135
        test_portion = 1 / 7
        
    else:
        SEED = 110
        N = len(X_data)
        test_portion = 0.2 # e.g. 0.2 means 20% test and 80% train
    
    init_type = 'left'
    portion = Fraction(test_portion).limit_denominator()
    denominator = portion.denominator
    numerator = portion.numerator


    if platform.system() == 'Windows':
        base_path = r'E:\BSNN'
    else:
        home = os.path.expanduser("~")
        base_path = os.path.join(home, f"BSNN")

    num_epochs = 50
    folder_path = os.path.join(base_path, f"{data_type}_test_portion_{numerator}_of_{denominator}")


    
    n_iterations = 5
    max_mode = 6
    photon_range = range(1,2,1)
    for num_photons in photon_range:
        modes_range = range(max_mode,max_mode+1,1)
        depth_range = modes_range
        for modes in modes_range:
            for depth in depth_range:
                print('N=',N)
                # if depth != modes:
                #     continue
                # else:
                main(N, test_portion, num_epochs, n_iterations, data_type,
                    folder_path, modes, depth, X_data, y_data, X_train, X_test, y_train, y_test, num_photons,numerator, denominator)



print("Done!!!")

